<a href="https://colab.research.google.com/github/ikwak2/4Dgradcam/blob/main/21_min/min_analysis250604.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

주제: 식욕 억제제 처방을 받은 환자들 중 객관적 and 주관적 후각 or 미각 장애 환자들은 얼마나 되며, 어떤 임상적 특징을 가지고 있을까?

1. Descriptive data: 실제 검사 상 후각, 미각 기능 이상을 보이는 환자들에 대하여 (통계량 뽑을 때 참여 취소 등 제외여부 확인, 참여취소, F/U loss 등)
1-1. 객관적 검사상 후각 기능 저하를 보인 경우와 정상인 경우에 대하여 유의한 factor 가 있는지 통계 분석
-> 그루핑을 BX, BY로 하는 것일지? 후각은 괜찮은데, BY의 경우 미각 이상 환자는 1명밖에 없어서 미각 이상 관련 분석은 샘플 수 부족으로 어려움.. (기술통계 table 작성은 가능하지만, 통계적 비교는 불가능)


2. VAS score 1점 이상 기준으로 주관적 후각 이상을 호소한 경우 (파란 글씨 표기 환자 )에 대한 descriptive data
 (나머지 검사 항목에 대해서)
2-1. VAS score 1 점 이상 기준 (파란 글씨 표기 환자)으로 주관적 후각 이상을 호소한 경우와 주관적 후각 이상이 없는 환자에 대하여 유의한 factor 가 있는지 통계 분석

3. 후각 VAS score (BW 열) 와 비례 혹은 반비례하는 임상적 인자가 있는지 통계 분석
(VAS score 가 높을수록 주관적 후각 불편감이 있는 것)
	종속변수를 VAS 로 두고 다중선형회귀 고려
	TDI score 나 W행도 살펴보기

4. Parosmia 만 따로 분석?? (BV, 이상한 냄새를 느끼는 주관적 지표)  
=> 이건 각각 통계량 뽑을 때 점수분포 살펴봄  

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
system("gdown --id 17aqy06YOtRF7eGsxcqs1JDnT9Ldcbmdp")
data = read_csv("min250520.csv")

Rows: 70 Columns: 77
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): id_num, reginum, weak, Sex, Smoking, medication, nose_function, ta...
dbl (68): ENTdate, Age, Height, weight, BMI, SM, bodyfat, BMR, waist_circum,...
lgl  (1): etc

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
data[1:3,]

id_num,ENTdate,reginum,weak,Sex,Age,Smoking,Height,weight,BMI,⋯,a3-P,a5-P,sumQOD-P,P,sumVAS,nose_function,taste_function,Phentermine_Naltrexone,Orlistat,Liraglutide_Semaglutide
<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
OOG1,20240715,00897871,NA,F,45,none,165.0,64.4,23.7,⋯,2,0,7,58.33333,0,normosmia,normal,1,1,0
OOG2,20240715,01724462,NA,F,48,none,156.6,58.9,24.0,⋯,0,0,0,0.00000,0,normosmia,normal,1,1,0
OOG3,20240715,01722868,NA,M,28,none,173.2,98.1,32.7,⋯,0,0,0,0.00000,0,normosmia,normal,1,1,0


In [4]:
# data[,'BMI'] 행의 값이 23미만: 정상 (1)
# - 23이상, 25미만: 과체중 (2)
# - 25이상, 30미만: 1단계 비만 (3)
# - 30이상, 35미만: 2단계 비만 (4)
# - 35이상 : 3단계 비만 (5)  으로 코딩해줘

data <- data |>
  mutate(BMI_category = case_when(
    BMI < 23 ~ 1,
    BMI >= 23 & BMI < 25 ~ 2,
    BMI >= 25 & BMI < 30 ~ 3,
    BMI >= 30 & BMI < 35 ~ 4,
    BMI >= 35 ~ 5
  ))

data[1:3, c('BMI', 'BMI_category')]

BMI,BMI_category
<dbl>,<dbl>
23.7,2
24.0,2
32.7,4


In [5]:
# BMI_category2 는 '- 23미만: 정상 (1)
# - 23이상, 25미만: 과체중 (2)
# - 25이상, 비만 (3) 로 만들어줘

data <- data |>
  mutate(BMI_category2 = case_when(
    BMI < 23 ~ 1,
    BMI >= 23 & BMI < 25 ~ 2,
    BMI >= 25 ~ 3
  ))

data[1:3, c('BMI', 'BMI_category', 'BMI_category2')]

BMI,BMI_category,BMI_category2
<dbl>,<dbl>,<dbl>
23.7,2,2
24.0,2,2
32.7,4,3


In [6]:
# BMI_category2 는 '- 23미만: 정상 (1)
# - 23이상, 25미만: 과체중 (2)
# - 25이상, 비만 (3) 로 만들어줘

data <- data |>
  mutate(BMI_category3 = case_when(
    BMI < 25 ~ 1,
    BMI >= 25 ~ 2
  ))

data[1:3, c('BMI', 'BMI_category', 'BMI_category2', 'BMI_category3')]

BMI,BMI_category,BMI_category2,BMI_category3
<dbl>,<dbl>,<dbl>,<dbl>
23.7,2,2,1
24.0,2,2,1
32.7,4,3,2


In [7]:
install.packages("moonBook")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘insight’, ‘datawizard’, ‘sjlabelled’, ‘nortest’, ‘sjmisc’




In [8]:
library(moonBook)

In [9]:
colnames(data)

[1] "id_num"                  "ENTdate"                
 [3] "reginum"                 "weak"                   
 [5] "Sex"                     "Age"                    
 [7] "Smoking"                 "Height"                 
 [9] "weight"                  "BMI"                    
[11] "SM"                      "bodyfat"                
[13] "BMR"                     "waist_circum"           
[15] "Threshold"               "Discrimination"         
[17] "Identification"          "TDI"                    
[19] "tasteThreshold"          "tasteIdentification"    
[21] "WBC"                     "hs-CRP"                 
[23] "fasting_glucose"         "uric_acid"              
[25] "AST"                     "ALT"                    
[27] "insulin"                 "T_cholesterol"          
[29] "triglyceride"            "HDLcholesterol"         
[31] "LDLcholesterol"          "cortisol"               
[33] "freeT4"                  "TSH"                    
[35] "medication"              "etc"                    
[37] "a1-LQ"                   "a4-LQ"                  
[39] "a11-LQ"                  "a13-LQ"                 
[41] "a15-LQ"                  "a19-LQ"                 
[43] "a22-LQ"                  "a26-LQ"                 
[45] "a27-LQ"                  "a28-LQ"                 
[47] "a33-LQ"                  "a34-LQ"                 
[49] "a37-LQ"                  "a39-LQ"                 
[51] "a42-LQ"                  "a49-LQ"                 
[53] "a50-LQ"                  "a32-LQ"                 
[55] "a35-LQ"                  "sumQOD-LQ"              
[57] "LQ"                      "a14-S"                  
[59] "a17-S"                   "a23-S"                  
[61] "a31-S"                   "a36-S"                  
[63] "a48-S"                   "sumQOD-S"               
[65] "S"                       "a1-P"                   
[67] "a2-P"                    "a3-P"                   
[69] "a5-P"                    "sumQOD-P"               
[71] "P"                       "sumVAS"                 
[73] "nose_function"           "taste_function"         
[75] "Phentermine_Naltrexone"  "Orlistat"               
[77] "Liraglutide_Semaglutide" "BMI_category"           
[79] "BMI_category2"           "BMI_category3"

In [10]:
data$subjective_nose = as.numeric(data[,'sumVAS'] >= 1)
data$Smoking <- as.numeric(ifelse(data$Smoking == 'none', 0, data$Smoking))

In [11]:
data$subjective_nose

[1] 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 0 0 1 1 1 0 1 0 1 1 1 0 1 1 1 1 1
[39] 0 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 1 0 1 1 1 1 1 0 0 0 0 0 1 1 1 1

## Descriptive data: 실제 검사 상 후각, 미각 기능 이상을 보이는 환자들에 대하여

###1-1. 객관적 검사상 후각 기능 저하를 보인 경우와 정상인 경우에 대하여 유의한 factor 가 있는지 통계 분석

In [12]:
?mytable

In [13]:
mytable(nose_function ~ Phentermine_Naltrexone + Orlistat + Liraglutide_Semaglutide+subjective_nose + P, data=data, digits=2, catMethod=0)


        Descriptive Statistics by 'nose_function'        
—————————————————————————————————————————————————————————— 
                           hyposmia      normosmia     p  
                             (N=5)        (N=65)    
—————————————————————————————————————————————————————————— 
 Phentermine_Naltrexone                              0.375
   - 0                    1 (20.00%)     5 (7.81%)        
   - 1                    4 (80.00%)    59 (92.19%)       
 Orlistat                                            1.000
   - 0                     0 ( 0.0%)     1 (1.54%)        
   - 1                    5 (100.00%)   64 (98.46%)       
 Liraglutide_Semaglutide                             0.508
   - 0                    4 (80.00%)    57 (87.69%)       
   - 1                    1 (20.00%)    8 (12.31%)        
 subjective_nose                                     0.358
   - 0                    4 (80.00%)    32 (49.23%)       
   - 1                    1 (20.00%)    33 (50.77%)       
 

VAS 가 1보다 큰 주관적 후각이상이 후각이상 그룹에서는 오히려 적었고, 정상인 그룹은 VAS 1이상 분포가 반반..

In [14]:
str(data$Identification)

 num [1:70] 12 11 12 12 11 12 11 12 12 12 ...


In [15]:
install.packages("tableone")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gtools’, ‘minqa’, ‘numDeriv’, ‘mitools’, ‘RcppArmadillo’, ‘proxy’, ‘gdata’, ‘survey’, ‘e1071’, ‘zoo’, ‘gmodels’, ‘labelled’




In [16]:
library(tableone)


In [17]:
CreateTableOne(vars = c("Age", "Height", "weight", "BMI", "SM", "bodyfat", "BMR", "waist_circum", "Threshold", "Discrimination", "Identification", "TDI", "tasteThreshold", "tasteIdentification" , "WBC", "hs-CRP", "fasting_glucose", "uric_acid", "AST", "ALT", "insulin", "T_cholesterol", "triglyceride", "HDLcholesterol", "LDLcholesterol", "cortisol", "freeT4", "TSH"), strata = "nose_function", data = data)

                                 Stratified by nose_function
                                  hyposmia         normosmia        p      test
  n                                     5               65                     
  Age (mean (SD))                   46.60 (15.82)    41.25 (10.50)   0.293     
  Height (mean (SD))               158.76 (10.53)   163.30 (7.77)    0.223     
  weight (mean (SD))                89.64 (19.24)    80.29 (19.99)   0.316     
  BMI (mean (SD))                   35.36 (5.47)     29.89 (5.70)    0.042     
  SM (mean (SD))                    29.54 (9.17)     27.05 (6.30)    0.412     
  bodyfat (mean (SD))               41.58 (6.17)     38.10 (6.28)    0.236     
  BMR (mean (SD))                 1506.00 (310.92) 1430.58 (225.81)  0.485     
  waist_circum (mean (SD))         107.68 (11.61)    97.09 (14.63)   0.120     
  Threshold (mean (SD))              2.80 (0.91)      4.02 (1.01)    0.011     
  Discrimination (mean (SD))         6.80 (1.30)      9.14 

연속형 변수

In [18]:
CreateTableOne(vars = c("Age", "Height", "weight", "BMI", "SM", "bodyfat", "BMR", "waist_circum", "Threshold", "Discrimination", "Identification", "TDI", "tasteThreshold", "tasteIdentification" , "WBC", "hs-CRP", "fasting_glucose", "uric_acid", "AST", "ALT", "insulin", "T_cholesterol", "triglyceride", "HDLcholesterol", "LDLcholesterol", "cortisol", "freeT4", "TSH", "P"), strata = "nose_function", data = data)

                                 Stratified by nose_function
                                  hyposmia         normosmia        p      test
  n                                     5               65                     
  Age (mean (SD))                   46.60 (15.82)    41.25 (10.50)   0.293     
  Height (mean (SD))               158.76 (10.53)   163.30 (7.77)    0.223     
  weight (mean (SD))                89.64 (19.24)    80.29 (19.99)   0.316     
  BMI (mean (SD))                   35.36 (5.47)     29.89 (5.70)    0.042     
  SM (mean (SD))                    29.54 (9.17)     27.05 (6.30)    0.412     
  bodyfat (mean (SD))               41.58 (6.17)     38.10 (6.28)    0.236     
  BMR (mean (SD))                 1506.00 (310.92) 1430.58 (225.81)  0.485     
  waist_circum (mean (SD))         107.68 (11.61)    97.09 (14.63)   0.120     
  Threshold (mean (SD))              2.80 (0.91)      4.02 (1.01)    0.011     
  Discrimination (mean (SD))         6.80 (1.30)      9.14 

In [19]:
mytable(nose_function ~ Sex + Smoking + BMI_category + BMI_category2 + Phentermine_Naltrexone + Orlistat + Liraglutide_Semaglutide+subjective_nose  , data=data, digits=2, catMethod=0)


      Descriptive Statistics by 'nose_function'      
—————————————————————————————————————————————————————— 
                          hyposmia    normosmia    p  
                            (N=5)      (N=65)   
—————————————————————————————————————————————————————— 
 Sex                                             0.290
   - F                   3 (60.00%)  52 (80.00%)      
   - M                   2 (40.00%)  13 (20.00%)      
 Smoking                                         1.000
   - 0                   5 (100.00%) 61 (93.85%)      
   - 1                    0 ( 0.0%)   4 (6.15%)       
 BMI_category                                    0.560
   - 1                    0 ( 0.0%)   4 (6.15%)       
   - 2                    0 ( 0.0%)  7 (10.77%)       
   - 3                   1 (20.00%)  25 (38.46%)      
   - 4                   2 (40.00%)  20 (30.77%)      
   - 5                   2 (40.00%)  9 (13.85%)       
 BMI_category2                                   1.000
   - 1        

2. VAS score 1점 이상 기준으로 주관적 후각 이상을 호소한 경우 (파란 글씨 표기 환자 )에 대한 descriptive data
 (나머지 검사 항목에 대해서)
2-1. VAS score 1 점 이상 기준 (파란 글씨 표기 환자)으로 주관적 후각 이상을 호소한 경우와 주관적 후각 이상이 없는 환자에 대하여 유의한 factor 가 있는지 통계 분석

In [20]:
CreateTableOne(vars = c("Age", "Height", "weight", "BMI", "SM", "bodyfat", "BMR", "waist_circum", "Threshold", "Discrimination", "Identification", "TDI", "tasteThreshold", "tasteIdentification" , "WBC", "hs-CRP", "fasting_glucose", "uric_acid", "AST", "ALT", "insulin", "T_cholesterol", "triglyceride", "HDLcholesterol", "LDLcholesterol", "cortisol", "freeT4", "TSH", "P"), strata = "subjective_nose", data = data)

                                 Stratified by subjective_nose
                                  0                1                p      test
  n                                    36               34                     
  Age (mean (SD))                   38.86 (10.55)    44.56 (10.62)   0.028     
  Height (mean (SD))               164.01 (7.53)    161.89 (8.42)    0.272     
  weight (mean (SD))                82.38 (22.33)    79.45 (17.29)   0.543     
  BMI (mean (SD))                   30.31 (6.02)     30.25 (5.69)    0.964     
  SM (mean (SD))                    27.80 (6.72)     26.61 (6.27)    0.448     
  bodyfat (mean (SD))               38.19 (5.46)     38.50 (7.16)    0.839     
  BMR (mean (SD))                 1455.25 (240.37) 1415.56 (222.03)  0.476     
  waist_circum (mean (SD))          98.00 (14.79)    97.69 (14.68)   0.931     
  Threshold (mean (SD))              4.14 (1.20)      3.71 (0.81)    0.083     
  Discrimination (mean (SD))         8.81 (1.33)      9.1

In [21]:
mytable(subjective_nose ~ Sex + Smoking + BMI_category + BMI_category2 + Phentermine_Naltrexone + Orlistat + Liraglutide_Semaglutide+nose_function  , data=data, digits=2, catMethod=0)


      Descriptive Statistics by 'subjective_nose'     
——————————————————————————————————————————————————————— 
                              0            1        p  
                            (N=36)      (N=34)   
——————————————————————————————————————————————————————— 
 Sex                                              0.647
   - F                   27 (75.00%)  28 (82.35%)      
   - M                    9 (25.00%)  6 (17.65%)       
 Smoking                                          0.615
   - 0                   33 (91.67%)  33 (97.06%)      
   - 1                    3 (8.33%)    1 (2.94%)       
 BMI_category                                     0.780
   - 1                    1 (2.78%)    3 (8.82%)       
   - 2                    4 (11.11%)   3 (8.82%)       
   - 3                   15 (41.67%)  11 (32.35%)      
   - 4                   10 (27.78%)  12 (35.29%)      
   - 5                    6 (16.67%)  5 (14.71%)       
 BMI_category2                                    0.

BMI 의 경우 0 인게 생겨서, 카테고리할 경우 분석이 안될 수 있음

## 처방 별로 nose function 과 관계?

In [22]:
mytable(Phentermine_Naltrexone ~ nose_function + subjective_nose + P, data=data, digits=2, catMethod=0)


Descriptive Statistics by 'Phentermine_Naltrexone'
—————————————————————————————————————————————————— 
                       0             1         p  
                     (N=6)        (N=63)    
—————————————————————————————————————————————————— 
 nose_function                               0.375
   - hyposmia     1 (16.67%)     4 (6.35%)        
   - normosmia    5 (83.33%)    59 (93.65%)       
 subjective_nose                             1.000
   - 0            3 (50.00%)    33 (52.38%)       
   - 1            3 (50.00%)    30 (47.62%)       
 P               22.22 ± 12.55 20.63 ± 20.29 0.852
—————————————————————————————————————————————————— 

In [23]:
mytable(Orlistat ~ nose_function + subjective_nose + P, data=data, digits=2, catMethod=0)

Error in var.test.default(x = DATA[[1L]], y = DATA[[2L]], ...) : 
  not enough 'x' observations



      Descriptive Statistics by 'Orlistat'      
————————————————————————————————————————————————— 
                      0             1         p  
                    (N=1)        (N=69)    
————————————————————————————————————————————————— 
 nose_function                              1.000
   - hyposmia      0 ( 0.0%)    5 (7.25%)        
   - normosmia   1 (100.00%)   64 (92.75%)       
 subjective_nose                            0.486
   - 0             0 ( 0.0%)   36 (52.17%)       
   - 1           1 (100.00%)   33 (47.83%)       
 P               16.67 ±   NA 20.53 ± 19.83      
————————————————————————————————————————————————— 

In [24]:
mytable(Liraglutide_Semaglutide ~ nose_function +subjective_nose + P , data=data, digits=2, catMethod=0)


Descriptive Statistics by 'Liraglutide_Semaglutide'
—————————————————————————————————————————————————— 
                       0             1         p  
                    (N=61)         (N=9)    
—————————————————————————————————————————————————— 
 nose_function                               0.508
   - hyposmia      4 (6.56%)    1 (11.11%)        
   - normosmia    57 (93.44%)   8 (88.89%)        
 subjective_nose                             0.300
   - 0            33 (54.10%)   3 (33.33%)        
   - 1            28 (45.90%)   6 (66.67%)        
 P               20.08 ± 19.74 23.15 ± 20.32 0.666
—————————————————————————————————————————————————— 

In [25]:
sum(data[,'sumVAS'] >= 1)

[1] 34

In [27]:
dim(data)

[1] 70 81

3. 후각 VAS score (BW 열) 와 비례 혹은 반비례하는 임상적 인자가 있는지 통계 분석
(VAS score 가 높을수록 주관적 후각 불편감이 있는 것)
	종속변수를 VAS 로 두고 다중선형회귀 고려
	TDI score 나 W행도 살펴보기

In [29]:
colnames(data)

[1] "id_num"                  "ENTdate"                
 [3] "reginum"                 "weak"                   
 [5] "Sex"                     "Age"                    
 [7] "Smoking"                 "Height"                 
 [9] "weight"                  "BMI"                    
[11] "SM"                      "bodyfat"                
[13] "BMR"                     "waist_circum"           
[15] "Threshold"               "Discrimination"         
[17] "Identification"          "TDI"                    
[19] "tasteThreshold"          "tasteIdentification"    
[21] "WBC"                     "hs-CRP"                 
[23] "fasting_glucose"         "uric_acid"              
[25] "AST"                     "ALT"                    
[27] "insulin"                 "T_cholesterol"          
[29] "triglyceride"            "HDLcholesterol"         
[31] "LDLcholesterol"          "cortisol"               
[33] "freeT4"                  "TSH"                    
[35] "medication"              "etc"                    
[37] "a1-LQ"                   "a4-LQ"                  
[39] "a11-LQ"                  "a13-LQ"                 
[41] "a15-LQ"                  "a19-LQ"                 
[43] "a22-LQ"                  "a26-LQ"                 
[45] "a27-LQ"                  "a28-LQ"                 
[47] "a33-LQ"                  "a34-LQ"                 
[49] "a37-LQ"                  "a39-LQ"                 
[51] "a42-LQ"                  "a49-LQ"                 
[53] "a50-LQ"                  "a32-LQ"                 
[55] "a35-LQ"                  "sumQOD-LQ"              
[57] "LQ"                      "a14-S"                  
[59] "a17-S"                   "a23-S"                  
[61] "a31-S"                   "a36-S"                  
[63] "a48-S"                   "sumQOD-S"               
[65] "S"                       "a1-P"                   
[67] "a2-P"                    "a3-P"                   
[69] "a5-P"                    "sumQOD-P"               
[71] "P"                       "sumVAS"                 
[73] "nose_function"           "taste_function"         
[75] "Phentermine_Naltrexone"  "Orlistat"               
[77] "Liraglutide_Semaglutide" "BMI_category"           
[79] "BMI_category2"           "BMI_category3"          
[81] "subjective_nose"

In [30]:
cor(data[,c('Threshold','Discrimination','Identification','TDI','tasteThreshold','tasteIdentification',"sumVAS")])

,Threshold,Discrimination,Identification,TDI,tasteThreshold,tasteIdentification,sumVAS
Threshold,1.00000000,-0.02120172,-0.10319839,0.4925610,0.13865892,0.26298317,-0.03633920
Discrimination,-0.02120172,1.00000000,0.24924097,0.7996778,0.27807670,0.11115736,0.18059236
Identification,-0.10319839,0.24924097,1.00000000,0.4552845,0.09591175,0.02122877,0.10377817
TDI,0.49256096,0.79967780,0.45528448,1.0000000,0.30948998,0.23057645,0.14615891
tasteThreshold,0.13865892,0.27807670,0.09591175,0.3094900,1.00000000,0.52960597,0.03651750
tasteIdentification,0.26298317,0.11115736,0.02122877,0.2305764,0.52960597,1.00000000,-0.05975038
sumVAS,-0.03633920,0.18059236,0.10377817,0.1461589,0.03651750,-0.05975038,1.00000000


In [31]:
('sumVAS','Age', 'Sex', 'Height', 'weight', 'BMI', 'SM', 'bodyfat', 'BMR', 'waist_circum', 'Threshold', 'Discrimination', 'Identification', 'TDI', 'tasteThreshold', 'tasteIdentification' , 'WBC', 'hs-CRP', 'fasting_glucose', 'uric_acid', 'AST', 'ALT', 'insulin', 'T_cholesterol', 'triglyceride', 'HDLcholesterol', 'LDLcholesterol', 'cortisol', 'freeT4', 'TSH', 'Phentermine_Naltrexone', 'Orlistat', 'Liraglutide_Semaglutide', 'P')])


ERROR: Error in parse(text = input): <text>:1:10: unexpected ','
1: ('sumVAS',
             ^


In [33]:
colnames(data)

[1] "id_num"                  "ENTdate"                
 [3] "reginum"                 "weak"                   
 [5] "Sex"                     "Age"                    
 [7] "Smoking"                 "Height"                 
 [9] "weight"                  "BMI"                    
[11] "SM"                      "bodyfat"                
[13] "BMR"                     "waist_circum"           
[15] "Threshold"               "Discrimination"         
[17] "Identification"          "TDI"                    
[19] "tasteThreshold"          "tasteIdentification"    
[21] "WBC"                     "hs-CRP"                 
[23] "fasting_glucose"         "uric_acid"              
[25] "AST"                     "ALT"                    
[27] "insulin"                 "T_cholesterol"          
[29] "triglyceride"            "HDLcholesterol"         
[31] "LDLcholesterol"          "cortisol"               
[33] "freeT4"                  "TSH"                    
[35] "medication"              "etc"                    
[37] "a1-LQ"                   "a4-LQ"                  
[39] "a11-LQ"                  "a13-LQ"                 
[41] "a15-LQ"                  "a19-LQ"                 
[43] "a22-LQ"                  "a26-LQ"                 
[45] "a27-LQ"                  "a28-LQ"                 
[47] "a33-LQ"                  "a34-LQ"                 
[49] "a37-LQ"                  "a39-LQ"                 
[51] "a42-LQ"                  "a49-LQ"                 
[53] "a50-LQ"                  "a32-LQ"                 
[55] "a35-LQ"                  "sumQOD-LQ"              
[57] "LQ"                      "a14-S"                  
[59] "a17-S"                   "a23-S"                  
[61] "a31-S"                   "a36-S"                  
[63] "a48-S"                   "sumQOD-S"               
[65] "S"                       "a1-P"                   
[67] "a2-P"                    "a3-P"                   
[69] "a5-P"                    "sumQOD-P"               
[71] "P"                       "sumVAS"                 
[73] "nose_function"           "taste_function"         
[75] "Phentermine_Naltrexone"  "Orlistat"               
[77] "Liraglutide_Semaglutide" "BMI_category"           
[79] "BMI_category2"           "BMI_category3"          
[81] "subjective_nose"

In [36]:
colnames(data)[5:34]

[1] "Sex"                 "Age"                 "Smoking"            
 [4] "Height"              "weight"              "BMI"                
 [7] "SM"                  "bodyfat"             "BMR"                
[10] "waist_circum"        "Threshold"           "Discrimination"     
[13] "Identification"      "TDI"                 "tasteThreshold"     
[16] "tasteIdentification" "WBC"                 "hs-CRP"             
[19] "fasting_glucose"     "uric_acid"           "AST"                
[22] "ALT"                 "insulin"             "T_cholesterol"      
[25] "triglyceride"        "HDLcholesterol"      "LDLcholesterol"     
[28] "cortisol"            "freeT4"              "TSH"

In [37]:
colnames(data)[c(71,72, 75,76,77)]

[1] "P"                       "sumVAS"                 
[3] "Phentermine_Naltrexone"  "Orlistat"               
[5] "Liraglutide_Semaglutide"

In [38]:
dataNew = data[,c(5:34, 71,72 ) ]

In [39]:

dataNew <- na.omit(dataNew)

dim(dataNew)

[1] 68 32

In [40]:
dataNew2 = data[,c(5:34, 71,73 ) ]

In [45]:
dataNew3 = data[,c(5:34, 71,72,73 ) ]

In [41]:
model <- lm(sumVAS ~ .  , data = dataNew)

# 후진 제거법 (backward elimination)
step_model <- step(model, direction = "backward")

# 모델 요약 출력
summary(step_model)

Start:  AIC=319.61
sumVAS ~ Sex + Age + Smoking + Height + weight + BMI + SM + bodyfat + 
    BMR + waist_circum + Threshold + Discrimination + Identification + 
    TDI + tasteThreshold + tasteIdentification + WBC + `hs-CRP` + 
    fasting_glucose + uric_acid + AST + ALT + insulin + T_cholesterol + 
    triglyceride + HDLcholesterol + LDLcholesterol + cortisol + 
    freeT4 + TSH + P


Step:  AIC=319.61
sumVAS ~ Sex + Age + Smoking + Height + weight + BMI + SM + bodyfat + 
    BMR + waist_circum + Threshold + Discrimination + Identification + 
    tasteThreshold + tasteIdentification + WBC + `hs-CRP` + fasting_glucose + 
    uric_acid + AST + ALT + insulin + T_cholesterol + triglyceride + 
    HDLcholesterol + LDLcholesterol + cortisol + freeT4 + TSH + 
    P

                      Df Sum of Sq    RSS    AIC
- Age                  1      0.06 3004.6 317.61
- Sex                  1      0.08 3004.6 317.61
- tasteThreshold       1      0.96 3005.5 317.63
- WBC                  1      1.


Call:
lm(formula = sumVAS ~ BMI + waist_circum + tasteIdentification + 
    fasting_glucose + ALT + triglyceride + cortisol + TSH + P, 
    data = dataNew)

Residuals:
    Min      1Q  Median      3Q     Max 
-13.222  -5.914  -1.195   4.599  24.921 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)   
(Intercept)          2.22965   12.43379   0.179  0.85831   
BMI                 -1.62863    0.54072  -3.012  0.00384 **
waist_circum         0.57674    0.22396   2.575  0.01259 * 
tasteIdentification -0.46900    0.32328  -1.451  0.15223   
fasting_glucose      0.06624    0.04040   1.640  0.10650   
ALT                 -0.09642    0.03713  -2.597  0.01190 * 
triglyceride        -0.02313    0.01533  -1.508  0.13690   
cortisol            -0.47132    0.33183  -1.420  0.16086   
TSH                  1.92105    0.69102   2.780  0.00732 **
P                    0.13493    0.05191   2.599  0.01183 * 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residu

In [42]:
library(MASS)
model <- lm(sumVAS ~ .  , data = dataNew) %>%
              stepAIC(trace = FALSE)
summary(model)


Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select





Call:
lm(formula = sumVAS ~ BMI + waist_circum + tasteIdentification + 
    fasting_glucose + ALT + triglyceride + cortisol + TSH + P, 
    data = dataNew)

Residuals:
    Min      1Q  Median      3Q     Max 
-13.222  -5.914  -1.195   4.599  24.921 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)   
(Intercept)          2.22965   12.43379   0.179  0.85831   
BMI                 -1.62863    0.54072  -3.012  0.00384 **
waist_circum         0.57674    0.22396   2.575  0.01259 * 
tasteIdentification -0.46900    0.32328  -1.451  0.15223   
fasting_glucose      0.06624    0.04040   1.640  0.10650   
ALT                 -0.09642    0.03713  -2.597  0.01190 * 
triglyceride        -0.02313    0.01533  -1.508  0.13690   
cortisol            -0.47132    0.33183  -1.420  0.16086   
TSH                  1.92105    0.69102   2.780  0.00732 **
P                    0.13493    0.05191   2.599  0.01183 * 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residu

In [43]:
colSums(is.na(data))

id_num                 ENTdate                 reginum 
                      0                       0                       0 
                   weak                     Sex                     Age 
                     47                       0                       0 
                Smoking                  Height                  weight 
                      0                       0                       0 
                    BMI                      SM                 bodyfat 
                      0                       0                       0 
                    BMR            waist_circum               Threshold 
                      0                       0                       0 
         Discrimination          Identification                     TDI 
                      0                       0                       0 
         tasteThreshold     tasteIdentification                     WBC 
                      0                       0                       0 
                 hs-CRP         fasting_glucose               uric_acid 
                      1                       0                       1 
                    AST                     ALT                 insulin 
                      0                       0                       0 
          T_cholesterol            triglyceride          HDLcholesterol 
                      0                       0                       0 
         LDLcholesterol                cortisol                  freeT4 
                      0                       0                       0 
                    TSH              medication                     etc 
                      0                       0                      70 
                  a1-LQ                   a4-LQ                  a11-LQ 
                      0                       0                       0 
                 a13-LQ                  a15-LQ                  a19-LQ 
                      0                       0                       0 
                 a22-LQ                  a26-LQ                  a27-LQ 
                      0                       0                       0 
                 a28-LQ                  a33-LQ                  a34-LQ 
                      0                       0                       0 
                 a37-LQ                  a39-LQ                  a42-LQ 
                      0                       0                       0 
                 a49-LQ                  a50-LQ                  a32-LQ 
                      0                       0                       0 
                 a35-LQ               sumQOD-LQ                      LQ 
                      0                       0                       0 
                  a14-S                   a17-S                   a23-S 
                      0                       0                       0 
                  a31-S                   a36-S                   a48-S 
                      0                       0                       0 
               sumQOD-S                       S                    a1-P 
                      0                       0                       0 
                   a2-P                    a3-P                    a5-P 
                      0                       0                       0 
               sumQOD-P                       P                  sumVAS 
                      0                       0                       0 
          nose_function          taste_function  Phentermine_Naltrexone 
                      0                       0                       1 
               Orlistat Liraglutide_Semaglutide            BMI_category 
                      0                       0                       0 
          BMI_category2           BMI_category3         subjective_nose 
                      0                       0                       0

### 4. VAS 와 TDI 상관관계 없었음

->질문은 TDI (후각 정상 vs 후각 이상으로 나누지 말고) score 들과 유의하게 연관성을 보이는 인자들이 있는지?

TDI 점수, Threshold 점수, Discrimination 점수, Identification 점수 자체와 positive or negative 연관성 보이는 인자가 있는지 추가 분석 부탁드립니다.

### TDI 점수 등등을 Y 로 두고 multivariate regression 적합

In [46]:
dataNew3 <- na.omit(dataNew3)

In [47]:
colnames(dataNew3)

[1] "Sex"                 "Age"                 "Smoking"            
 [4] "Height"              "weight"              "BMI"                
 [7] "SM"                  "bodyfat"             "BMR"                
[10] "waist_circum"        "Threshold"           "Discrimination"     
[13] "Identification"      "TDI"                 "tasteThreshold"     
[16] "tasteIdentification" "WBC"                 "hs-CRP"             
[19] "fasting_glucose"     "uric_acid"           "AST"                
[22] "ALT"                 "insulin"             "T_cholesterol"      
[25] "triglyceride"        "HDLcholesterol"      "LDLcholesterol"     
[28] "cortisol"            "freeT4"              "TSH"                
[31] "P"                   "sumVAS"              "nose_function"

In [56]:
colnames(dataNew3)[-c(11:13)]

[1] "Sex"                 "Age"                 "Smoking"            
 [4] "Height"              "weight"              "BMI"                
 [7] "SM"                  "bodyfat"             "BMR"                
[10] "waist_circum"        "TDI"                 "tasteThreshold"     
[13] "tasteIdentification" "WBC"                 "hs-CRP"             
[16] "fasting_glucose"     "uric_acid"           "AST"                
[19] "ALT"                 "insulin"             "T_cholesterol"      
[22] "triglyceride"        "HDLcholesterol"      "LDLcholesterol"     
[25] "cortisol"            "freeT4"              "TSH"                
[28] "P"                   "sumVAS"              "nose_function"

#### TDI score 모델링

In [68]:
colnames(dataNew3)[33]

[1] "nose_function"

In [69]:
model <- lm(TDI ~ .  , data = dataNew3[,-c(11:13,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = TDI ~ Age + BMI + SM + bodyfat + BMR + tasteIdentification + 
    WBC + `hs-CRP` + AST + ALT + T_cholesterol + HDLcholesterol + 
    LDLcholesterol, data = dataNew3[, -c(11:13, 33)])

Residuals:
    Min      1Q  Median      3Q     Max 
-3.3450 -0.9104  0.1914  0.9902  2.7989 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)          7.944303   7.160644   1.109 0.272157    
Age                 -0.078335   0.021247  -3.687 0.000528 ***
BMI                 -0.271415   0.133163  -2.038 0.046439 *  
SM                  -0.938998   0.596733  -1.574 0.121429    
bodyfat              0.215271   0.094192   2.285 0.026237 *  
BMR                  0.031972   0.015790   2.025 0.047846 *  
tasteIdentification  0.094913   0.060818   1.561 0.124458    
WBC                 -0.309886   0.116021  -2.671 0.009975 ** 
`hs-CRP`            -0.095376   0.065163  -1.464 0.149086    
AST                  0.015422   0.009059   1.702 0.094424 .  
ALT   

In [70]:
model <- lm(Threshold ~ .  , data = dataNew3[,-c(12:14,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = Threshold ~ weight + SM + bodyfat + tasteIdentification + 
    WBC + uric_acid + AST + P, data = dataNew3[, -c(12:14, 33)])

Residuals:
     Min       1Q   Median       3Q      Max 
-1.81068 -0.47780 -0.06957  0.58190  1.74371 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)          8.250455   2.487111   3.317 0.001560 ** 
weight               0.078530   0.045702   1.718 0.090983 .  
SM                  -0.196663   0.128691  -1.528 0.131812    
bodyfat             -0.118520   0.067294  -1.761 0.083383 .  
tasteIdentification  0.087668   0.033396   2.625 0.011015 *  
WBC                 -0.146564   0.064222  -2.282 0.026102 *  
uric_acid           -0.268738   0.103725  -2.591 0.012045 *  
AST                  0.013901   0.003979   3.493 0.000912 ***
P                   -0.017613   0.005640  -3.123 0.002773 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.8534 on 59 degrees of 

In [71]:
model <- lm(Discrimination ~ .  , data = dataNew3[,-c(11,13,14,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = Discrimination ~ Age + Height + BMI + SM + bodyfat + 
    BMR + tasteThreshold + `hs-CRP` + T_cholesterol + HDLcholesterol + 
    LDLcholesterol + sumVAS, data = dataNew3[, -c(11, 13, 14, 
    33)])

Residuals:
     Min       1Q   Median       3Q      Max 
-2.53972 -0.59003 -0.02703  0.71554  1.95973 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)    15.37842    8.30688   1.851 0.069500 .  
Age            -0.07244    0.01814  -3.993 0.000195 ***
Height         -0.21011    0.10079  -2.085 0.041755 *  
BMI            -0.55999    0.20929  -2.676 0.009807 ** 
SM             -0.77906    0.58065  -1.342 0.185206    
bodyfat         0.32618    0.10443   3.124 0.002850 ** 
BMR             0.03827    0.02001   1.913 0.060957 .  
tasteThreshold  0.47274    0.26679   1.772 0.081941 .  
`hs-CRP`       -0.12235    0.04928  -2.483 0.016114 *  
T_cholesterol  -0.02513    0.01009  -2.491 0.015797 *  
HDLcholesterol  0.02405    0.01432   1.680 0.0

In [72]:
model <- lm(Identification ~ .  , data = dataNew3[,-c(11,12,14,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = Identification ~ Smoking + weight + BMI + bodyfat + 
    WBC + ALT + T_cholesterol, data = dataNew3[, -c(11, 12, 14, 
    33)])

Residuals:
    Min      1Q  Median      3Q     Max 
-1.8669 -0.2356  0.1288  0.3819  0.7266 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)   11.214300   0.617416  18.163   <2e-16 ***
Smoking        0.471751   0.314217   1.501   0.1385    
weight         0.021269   0.010432   2.039   0.0459 *  
BMI           -0.108467   0.043969  -2.467   0.0165 *  
bodyfat        0.044808   0.019481   2.300   0.0249 *  
WBC           -0.066162   0.041852  -1.581   0.1192    
ALT           -0.006467   0.002505  -2.582   0.0123 *  
T_cholesterol  0.004449   0.001891   2.353   0.0219 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.5598 on 60 degrees of freedom
Multiple R-squared:  0.3093,	Adjusted R-squared:  0.2287 
F-statistic: 3.839 on 7 and 60 DF,  p-value: 0.001644


In [63]:
colnames(dataNew3)

[1] "Sex"                 "Age"                 "Smoking"            
 [4] "Height"              "weight"              "BMI"                
 [7] "SM"                  "bodyfat"             "BMR"                
[10] "waist_circum"        "Threshold"           "Discrimination"     
[13] "Identification"      "TDI"                 "tasteThreshold"     
[16] "tasteIdentification" "WBC"                 "hs-CRP"             
[19] "fasting_glucose"     "uric_acid"           "AST"                
[22] "ALT"                 "insulin"             "T_cholesterol"      
[25] "triglyceride"        "HDLcholesterol"      "LDLcholesterol"     
[28] "cortisol"            "freeT4"              "TSH"                
[31] "P"                   "sumVAS"              "nose_function"

In [73]:
model <- lm(tasteIdentification ~ .  , data = dataNew3[,-c(15,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = tasteIdentification ~ Age + BMR + Threshold + uric_acid + 
    ALT + insulin + triglyceride + HDLcholesterol + cortisol + 
    P + sumVAS, data = dataNew3[, -c(15, 33)])

Residuals:
    Min      1Q  Median      3Q     Max 
-6.9801 -1.5787 -0.1144  1.4933  5.4722 

Coefficients:
                Estimate Std. Error t value Pr(>|t|)    
(Intercept)    28.869423   5.198707   5.553 8.01e-07 ***
Age            -0.090511   0.036989  -2.447   0.0176 *  
BMR            -0.004440   0.002139  -2.076   0.0425 *  
Threshold       0.921041   0.370709   2.485   0.0160 *  
uric_acid       0.496695   0.372767   1.332   0.1881    
ALT            -0.034869   0.013692  -2.547   0.0137 *  
insulin         0.015734   0.010622   1.481   0.1442    
triglyceride   -0.011725   0.005941  -1.974   0.0534 .  
HDLcholesterol -0.050225   0.033125  -1.516   0.1351    
cortisol       -0.267484   0.121361  -2.204   0.0316 *  
P               0.041010   0.021988   1.865   0.0674 .  
sumVAS         -0

In [74]:

model <- lm(tasteThreshold ~ .  , data = dataNew3[,-c(16,33)]) %>%
              stepAIC(trace = FALSE)
summary(model)


Call:
lm(formula = tasteThreshold ~ Sex + Height + Discrimination + 
    uric_acid + T_cholesterol + cortisol + freeT4, data = dataNew3[, 
    -c(16, 33)])

Residuals:
     Min       1Q   Median       3Q      Max 
-1.15610 -0.39467  0.06271  0.36178  0.94049 

Coefficients:
                Estimate Std. Error t value Pr(>|t|)   
(Intercept)     0.723078   1.944852   0.372   0.7114   
SexM           -0.801349   0.280693  -2.855   0.0059 **
Height          0.016789   0.011778   1.426   0.1592   
Discrimination  0.095706   0.047353   2.021   0.0477 * 
uric_acid       0.078300   0.056695   1.381   0.1724   
T_cholesterol   0.002369   0.001691   1.401   0.1663   
cortisol       -0.037744   0.019991  -1.888   0.0639 . 
freeT4          0.634873   0.427077   1.487   0.1424   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.5068 on 60 degrees of freedom
Multiple R-squared:  0.2641,	Adjusted R-squared:  0.1782 
F-statistic: 3.076 on 7 and 60 DF,  p

In [50]:
model <- lm(TDI ~ Sex + Age + Smoking + Height + weight + BMI + SM + bodyfat + BMR + waist_circum + WBC + 'hs-CRP' + fasting_glucose + uric_acid + AST + ALT + insulin + T_cholesterol + triglyceride + HDLcholesterol + LDLcholesterol + cortisol + freeT4 + TSH +P + sumVAS  , data = dataNew3) %>%
              stepAIC(trace = FALSE)
summary(model)

ERROR: Error in terms.formula(formula, data = data): invalid model formula in ExtractVars


In [44]:
# prompt: nose_function 을 y로 해서 logistic regression 적합

# nose_function을 y로 한 logistic regression 적합
# nose_function은 범주형 변수인지 확인 필요 (0 또는 1로 코딩되어 있다고 가정)
# 만약 연속형이라면 linear regression 사용

# nose_function 변수 확인 (0 또는 1로 되어있어야 함)
# unique(data$nose_function)

# 로지스틱 회귀 모델 적합
# nose_function ~ 나머지 변수들 (종속 변수와 독립 변수를 적절히 선택)
# 여기서는 모든 다른 변수를 독립 변수로 사용한다고 가정
# 종속 변수가 nose_function이고 독립 변수가 다른 모든 변수라고 가정
# 모델 적합 시 NA 값 처리가 필요할 수 있습니다.
model_logistic <- glm(nose_function ~ ., data = data[, -which(names(data) == "nose_function")], family = binomial)

# 모델 요약
summary(model_logistic)

# 필요한 경우 후진 제거법 또는 다른 변수 선택 방법 사용
# step_logistic <- step(model_logistic, direction = "backward")
# summary(step_logistic)


ERROR: Error in eval(predvars, data, env): object 'nose_function' not found


In [ ]:
dataNew2 <- na.omit(dataNew2)

dim(dataNew2)

In [ ]:
colnames(data)[14]

BMI group에 따라
1. 후각검사 상의 thredhold  discrimination  identification
2. 미각검사 상의 detectoon threshold, recognition threshold
3. 주관적 설문지 결과에 차이가 있는지 추가 분석 부탁드려도 될지요?



In [ ]:
mytable(nose_function ~ BMI + BMI_category + BMI_category2 , data=data, digits=2, catMethod=0)

In [ ]:
colnames(data)

In [ ]:
mytable(BMI_category ~  Threshold + Discrimination + Identification + TDI + tasteThreshold + tasteIdentification + nose_function, data=data, digits=2, catMethod=0)

In [ ]:
mytable(BMI_category2 ~  Threshold + Discrimination + Identification + TDI + tasteThreshold + tasteIdentification + nose_function, data=data, digits=2, catMethod=0)

In [ ]:
mytable(BMI_category3 ~  Threshold + Discrimination + Identification + TDI + tasteThreshold + tasteIdentification + nose_function, data=data, digits=2, catMethod=0)

BMI 25 기준으로 나누면 thrshold 에서는 차이가 있음